# Unit 1: Microsoft Foundry, first contact

This lecture was a tour, so this notebook is short. Its whole job is to prove
that the platform you watched me use in class is reachable from **your** code,
through the same lane switch you have used since Lecture 1.

**You do not need a Foundry account for this.** Every cell below runs on whatever
lane you already have configured. If that is the free GitHub lane, good. If you
set up your own Foundry deployment, even better, but it is optional.

## 1. Which lane are you actually on?

Print it. Half of all "it stopped working" reports are really "I am on a
different lane than I thought".

In [ ]:
from cse476.lanes import get_client, MODEL, describe

print(describe())
client = get_client()

## 2. The receipt

In the playground I showed you that every answer comes with a receipt: the model
name, the time, and the token count. You can read that same receipt from code.
The token count is the one that matters, because it is what you pay for.

In [ ]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "In one sentence, what is an AI agent?"}],
)

print(resp.choices[0].message.content)
print()
u = resp.usage
print(f"prompt tokens:     {u.prompt_tokens}")
print(f"completion tokens: {u.completion_tokens}")
print(f"total tokens:      {u.total_tokens}")

Notice how small that total is. Hold that number in your head for the next
cell.

## 3. The lesson from the playground, reproduced

In class, one short question cost 9436 tokens, because a web search tool was
switched on and its results were stuffed into the context. We are not going to
call a search tool here, but we can show the same *shape* of problem: the cost
is driven by what goes **in**, not by how short your question looks.

Watch the prompt token count climb as we prepend more context to the identical
question.

In [ ]:
question = "What is an AI agent?"

for pad_paragraphs in (0, 5, 20):
    # WHY: this padding stands in for the search results, documents, or long
    # history that a real tool or a long conversation would inject. The question
    # is identical every time. Only the context in front of it grows.
    context = "Background information you must consider. " * 40 * pad_paragraphs
    messages = [{"role": "user", "content": context + question}]

    r = client.chat.completions.create(model=MODEL, messages=messages)
    print(f"{pad_paragraphs:>3} paragraphs of context -> "
          f"{r.usage.prompt_tokens:>6} prompt tokens billed")

The question never changed. The bill did, a lot.

That is the whole point of the playground's 9436-token surprise: **you pay for
everything that reaches the model, including the parts you did not type and
cannot see in the chat window.** A tool that injects search results, a long
conversation you never trimmed, a giant document in the context, all of it is
billed, all of it is invisible in the message you see.

## 4. Deployment name versus model name

This one only bites Foundry users, but it is worth understanding even if you are
on the free lane, because it is the single most common Foundry error.

On Foundry, `MODEL` in your `.env` is the **deployment name** you chose
(`chat-demo`), **not** the underlying model name (`gpt-5-mini`). The lane hands
whatever is in `MODEL` straight to the API as the model to call.

In [ ]:
# This is all get_client does with your MODEL value: passes it through.
# On Foundry, the API resolves it as a deployment name.
# On GitHub or Groq, it resolves it as a model name.
# Same code, different meaning per lane. That is the abstraction working.
print(f"the value your code sends as 'model': {MODEL!r}")

If you are on Foundry and you set `MODEL=gpt-5-mini` instead of
`MODEL=chat-demo`, you get a 404 that reads as though the model does not exist.
It does exist. You just asked for it by the wrong name. **It is almost always
this.**

## Your turn

**1. Read your own receipt.** Run a question of your own and print the token
usage. Then make the question longer and watch the prompt tokens rise.

**2. Price a runaway.** In class I set my Foundry deployment's rate limit to
10,000 tokens per minute. If an uncapped agent looped flat out for one hour at
that rate, roughly how many tokens would it burn? Work it out. That number is
why the cap exists.

**3. Optional, Foundry only.** If you set up your own deployment, put
`PROVIDER=foundry` in your `.env`, run `python setup_check.py`, and confirm the
live call passes. Then switch back to `PROVIDER=github` for daily work. Foundry
is your projector lane, not your everyday one.

In [ ]:
# your work here
